In [9]:
import re
import random
import pandas as pd
from collections import defaultdict

class MarkovMon:
    def __init__(self):
        self.model = defaultdict(list)
        
        self.suffix_to_label = {
            "сан": "PAST", "сэн": "PAST", "сон": "PAST", "сөн": "PAST",
            "даг": "HABITUAL", "дэг": "HABITUAL", "дог": "HABITUAL", "дөг": "HABITUAL",
            "на": "FUTURE", "нэ": "FUTURE", "но": "FUTURE", "нө": "FUTURE",
        }

    def clean_sentence(self, text):
        text = re.sub(r'\b(OTHER|PAST|HABITUAL|FUTURE|other|past|habitual|future)\b', '', text)
        text = text.strip().rstrip('.,!?;:').strip()
        return text

    def train(self, sentences):
        for sentence in sentences:
            words = sentence.split()
            words = ["<START>"] + words + ["<END>"]
            for i in range(len(words) - 1):
                self.model[words[i]].append(words[i + 1])

    def detect_label(self, sentence):
        last_word = sentence.split()[-1]
        for suffix, label in self.suffix_to_label.items():
            if last_word.endswith(suffix):
                return label
        return None

    def generate_one(self, target_suffix, max_len=15, max_attempts=100):
        for _ in range(max_attempts):
            current = "<START>"
            result = []

            for _ in range(max_len):
                next_words = self.model.get(current, ["<END>"])
                next_word = random.choice(next_words)
                if next_word == "<END>":
                    break
                result.append(next_word)
                current = next_word

            if result and result[-1].endswith(target_suffix):
                label = self.suffix_to_label[target_suffix]
                return " ".join(result), label

        return None, None

    def label_sentence(self, sentence, tense_label):
        words = sentence.split()
        labels = ["Other"] * len(words)
        if words:
            labels[-1] = tense_label
        return words, labels

    def generate(self, per_suffix=3):
        results = []
        for suffix in self.suffix_to_label:
            count = 0
            attempts = 0
            while count < per_suffix and attempts < per_suffix * 50:
                sentence, label = self.generate_one(suffix)
                if sentence:
                    words, labels = self.label_sentence(sentence, label)
                    results.append((words, labels))
                    count += 1
                attempts += 1
        return results


# ---- LOAD ----
df = pd.read_csv("filtered_sentences.csv")

gen = MarkovMon()
sentences = df["sentence"].astype(str).apply(gen.clean_sentence).tolist()
sentences = [s for s in sentences if s]
print(f"Training on {len(sentences)} sentences.\n")

# ---- TRAIN + GENERATE ----
gen.train(sentences)

n = int(input("How many sentences per type? "))
outputs = gen.generate(per_suffix=n)

if not outputs:
    print("No sentences generated. The training sentences may not end with the target suffixes.")
else:
    prev_label = None
    for words, labels in outputs:
        sentence = " ".join(words)
        label_str = " ".join(labels)
        current_label = labels[-1]

        if prev_label is not None and current_label != prev_label:
            print()

        print(f"{sentence.capitalize()}, {label_str.upper()}")
        prev_label = current_label

Training on 436 sentences.

Тэр ажлаа дуусгасан, OTHER OTHER PAST
Хөдөө амьтдын амьдралд амжилттай дуусгасан, OTHER OTHER OTHER OTHER PAST
Тэр алдаа гаргасан, OTHER OTHER PAST
Тэр туршлагаас суралцсан, OTHER OTHER PAST
Тэр олон асуулт их зүйл сурсан, OTHER OTHER OTHER OTHER OTHER PAST
Тэр найзтайгаа уулзсан, OTHER OTHER PAST
Тэр шийдвэр гаргасан, OTHER OTHER PAST
Бид хамтдаа даван туулсан, OTHER OTHER OTHER PAST
Судалгаа шинэ зүйл ярьсан, OTHER OTHER OTHER PAST
Би учир шалтгаангүй дуралсан, OTHER OTHER OTHER PAST
Итгэл үнэмшилтэй хүмүүс амьдралд амжилттай дуусгасан, OTHER OTHER OTHER OTHER OTHER PAST
Бид хамт ажилласан, OTHER OTHER PAST
Тэр найзтайгаа уулзсан, OTHER OTHER PAST
Тэр туршлагаас суралцсан, OTHER OTHER PAST
Aхимаг насныхны хэмжээ багасан, OTHER OTHER OTHER PAST
Би өчигдөр ном уншсан, OTHER OTHER OTHER PAST
Хичээл зүтгэл үр дүнд нобелийн шагнал авсан, OTHER OTHER OTHER OTHER OTHER OTHER PAST
Бид төлөвлөгөө боловсруулсан, OTHER OTHER PAST
Судалгаа асуудлыг олон зүйл сурсан, O

In [10]:
# Save to CSV
if not outputs:
    print("No sentences generated. The training sentences may not end with the target suffixes.")
else:
    rows = []
    for words, labels in outputs:
        sentence = " ".join(words).capitalize()
        label_str = " ".join(labels).upper()
        tense = labels[-1].upper()
        rows.append({"sentence": sentence, "label": label_str})

    out_df = pd.DataFrame(rows)
    out_df.to_csv("generated_sentences.csv", index=False, encoding="utf-8-sig")
    print(f"Saved {len(rows)} sentences to 'generated_sentences.csv'.")

Saved 480 sentences to 'generated_sentences.csv'.
